# Double Pendulum Identification with RHONN

## Overview
Neural identification using three training algorithms:
- **EKF-RHONN**: Extended Kalman Filter
- **UKF-RHONN**: Unscented Kalman Filter  
- **PF-RHONN**: Particle Filter (neuron-specific parameters)

## Features
- RK4 discretization
- Non-Gaussian noise models
- NaN prevention strategies
- Comprehensive error metrics

## Execution
Run cells in order: Definitions → Simulation → Visualizations

In [1]:
# Neural Identifier Training - Double Pendulum
# Methods: EKF, UKF, Particle Filter
# Con características específicas por neurona

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1) True nonlinear system (Double Pendulum)
# ============================================================
def plant_dynamics(state, u):
    """
    Dynamics of a double pendulum with damping and external torques.
    state = [θ1, θ2, ω1, ω2] 
        θ1, θ2: angles (rad)
        ω1, ω2: angular velocities (rad/s)
    u = [τ1, τ2]: external torques on joints (N⋅m)
    """
    # Physical Parameters
    m1 = 1.0     # Mass of first pendulum (kg)
    m2 = 1.0     # Mass of second pendulum (kg)
    l1 = 1.0     # Length of first pendulum (m)
    l2 = 1.0     # Length of second pendulum (m)
    g = 9.81     # Gravity (m/s²)
    b1 = 0.1     # Damping coefficient for joint 1 (N⋅m⋅s)
    b2 = 0.1     # Damping coefficient for joint 2 (N⋅m⋅s)
    
    θ1, θ2, ω1, ω2 = state
    τ1, τ2 = u
    
    # Precompute trigonometric terms
    c12 = np.cos(θ1 - θ2)  # cos(θ1 - θ2)
    s12 = np.sin(θ1 - θ2)  # sin(θ1 - θ2)
    s1 = np.sin(θ1)         # sin(θ1)
    s2 = np.sin(θ2)         # sin(θ2)
    
    # Mass matrix elements
    M11 = (m1 + m2) * l1**2
    M12 = m2 * l1 * l2 * c12
    M21 = M12
    M22 = m2 * l2**2
    
    # Coriolis and centrifugal terms
    h = m2 * l1 * l2 * ω2**2 * s12
    C1 = h - (m1 + m2) * g * l1 * s1 - b1 * ω1 + τ1
    
    C2 = -m2 * l1 * l2 * ω1**2 * s12 - m2 * g * l2 * s2 - b2 * ω2 + τ2
    
    # Solve for accelerations using the mass matrix
    # [M11  M12] [α1]   [C1]
    # [M21  M22] [α2] = [C2]
    det_M = M11 * M22 - M12 * M21
    
    if abs(det_M) < 1e-10:
        det_M = 1e-10  # Prevent division by zero
    
    α1 = (M22 * C1 - M12 * C2) / det_M
    α2 = (-M21 * C1 + M11 * C2) / det_M
    
    # State derivatives
    θ1_dot = ω1
    θ2_dot = ω2
    ω1_dot = α1
    ω2_dot = α2
    
    return np.array([θ1_dot, θ2_dot, ω1_dot, ω2_dot])

def plant(x_k, u_k, dt=0.01, process_noise_std=1e-3):
    """
    RK4 integration step for better accuracy.
    """
    # RK4 para mayor precisión
    k1 = plant_dynamics(x_k, u_k)
    k2 = plant_dynamics(x_k + 0.5*dt*k1, u_k)
    k3 = plant_dynamics(x_k + 0.5*dt*k2, u_k)
    k4 = plant_dynamics(x_k + dt*k3, u_k)
    
    x_kp1 = x_k + (dt/6.0) * (k1 + 2*k2 + 2*k3 + k4)
    
    # Add small process noise
    noise = np.random.normal(0, process_noise_std, size=4)
    return x_kp1 #+ noise

# ============================================================
# 2) RHONN structure - CARACTERÍSTICAS POR NEURONA
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -50, 50) 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input, neuron_index):
    """
    Features for Double Pendulum - ESPECÍFICAS PARA CADA NEURONA.
    
    x_est = [θ1, θ2, ω1, ω2]
    u_input = [τ1, τ2]
    neuron_index: índice de la neurona (0=θ1, 1=θ2, 2=ω1, 3=ω2)
    """
    θ1, θ2, ω1, ω2 = x_est
    τ1, τ2 = u_input
    
    # Términos básicos sigmoidales
    s_θ1 = sigmoidal(θ1)
    s_θ2 = sigmoidal(θ2)
    s_ω1 = sigmoidal(ω1)
    s_ω2 = sigmoidal(ω2)
    
    # Términos trigonométricos (importantes para la dinámica del péndulo)
    # cos_θ1 = np.cos(θ1)
    # sin_θ1 = np.sin(θ1)
    # cos_θ2 = np.cos(θ2)
    # sin_θ2 = np.sin(θ2)
    # cos_θ12 = np.cos(θ1 - θ2)
    # sin_θ12 = np.sin(θ1 - θ2)
    
    # Comandos escalados
    s_τ1 = sigmoidal(τ1)
    s_τ2 = sigmoidal(τ2)
    
    # ========== CARACTERÍSTICAS ESPECÍFICAS POR NEURONA ==========
    
    if neuron_index == 0:  # Neurona para θ1 (ángulo del primer péndulo)
        # dθ1/dt = ω1
        return np.array([
            # s_ω1,                      # Velocidad angular (término principal)
            s_ω1**3,                   # Término cuadrático
            s_θ1**2,                      # Posición angular
            s_θ1 * s_ω1,              # Acoplamiento posición-velocidad
            # s_ω2,                      # Acoplamiento con segundo péndulo
            # cos_θ12,                   # Término de acoplamiento geométrico
        ])
    
    elif neuron_index == 1:  # Neurona para θ2 (ángulo del segundo péndulo)
        # dθ2/dt = ω2
        return np.array([
            s_ω2,                      # Velocidad angular (término principal)
            s_ω2**2,                   # Término cuadrático
            # s_θ2,                      # Posición angular
            s_θ2 * s_ω2,              # Acoplamiento posición-velocidad
            s_ω1,                      # Acoplamiento con primer péndulo
            # cos_θ12,                   # Término de acoplamiento geométrico
        ])
    
    elif neuron_index == 2:  # Neurona para ω1 (velocidad angular del primer péndulo)
        # dω1/dt = función compleja de θ1, θ2, ω1, ω2, τ1
        return np.array([
            s_ω1,                      # Estado actual
            s_ω1**2,                   # Término cuadrático (fricción)
            # sin_θ1,                    # Término gravitacional
            # sin_θ12,                   # Acoplamiento con segundo péndulo
            # s_ω2 * sin_θ12,           # Término de Coriolis
            s_τ1,                      # Torque externo
            # cos_θ12,                   # Acoplamiento geométrico
            s_ω1 * s_ω2,              # Interacción velocidades
        ])
    
    elif neuron_index == 3:  # Neurona para ω2 (velocidad angular del segundo péndulo)
        # dω2/dt = función compleja de θ1, θ2, ω1, ω2, τ2
        return np.array([
            s_ω2,                      # Estado actual
            s_ω2**2,                   # Término cuadrático (fricción)
            # sin_θ2,                    # Término gravitacional
            # sin_θ12,                   # Acoplamiento con primer péndulo
            # s_ω1 * sin_θ12,           # Término de Coriolis
            s_τ2,                      # Torque externo
            # cos_θ12,                   # Acoplamiento geométrico
            s_ω1 * s_ω2,              # Interacción velocidades
        ])
    
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# Función auxiliar para obtener el tamaño de características de cada neurona
def get_z_size(neuron_index):
    """Retorna el número de características para una neurona dada."""
    if neuron_index == 0:  # θ1
        return 3
    elif neuron_index == 1:  # θ2
        return 4
    elif neuron_index == 2:  # ω1
        return 4
    elif neuron_index == 3:  # ω2
        return 4
    else:
        raise ValueError(f"Índice de neurona inválido: {neuron_index}")

# ============================================================
# 3) Trainers (EKF, UKF, PF) - ADAPTADOS PARA MÚLTIPLES TAMAÑOS
# ============================================================

class Generic_RHONN_Trainer:
    """ Base class to handle the loop logic easily """
    def get_prediction(self, weights, x_k, u_k, neuron_idx):
        z = construct_z_vector(x_k, u_k, neuron_idx)
        return np.dot(weights, z)

class EKF_Trainer(Generic_RHONN_Trainer):
    '''
    eta = 0.9780
   P0 = 3.9958
   Q = 1.24e-03
   R = 2.46e-04
    '''
    def __init__(self, n_neurons, eta=1.0, P0=1.0, Q=1e-3, R=1e-5):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i))*P0 for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*Q for i in range(n_neurons)]
        self.R = R
        self.eta = eta

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            # Construir vector de características H_i (ecuación 8)
            z = construct_z_vector(x_k, u_k, i)
            H_i = z.reshape(-1, 1)
            
            # Error de identificación e_i(k) (ecuación 7)
            # e_i(k) = x_i(k) - X̂_i(k)
            x_hat_i = np.dot(self.weights[i], z)
            e_i = x_kp1[i] - x_hat_i
            
            # Ganancia de Kalman K_i(k) (ecuación 6)
            # K_i(k) = P_i(k) H_i(k) [R_i(k) + H_i(k) P_i(k) H_i(k)]^{-1}
            S = self.R + (H_i.T @ self.P[i] @ H_i)[0, 0]
            K_i = (self.P[i] @ H_i).flatten() / S
            
            # Actualización de pesos ω_i(k+1) (ecuación superior)
            # ω_i(k+1) = ω_i(k) + η_i K_i(k) e_i(k)
            self.weights[i] = self.weights[i] + self.eta * K_i * e_i
            
            # Actualización de covarianza P_i(k+1) (ecuación 6, tercera línea)
            # P_i(k+1) = P_i(k) - K_i(k) H_i(k) P_i(k) + Q_i(k)
            self.P[i] = self.P[i] - np.outer(K_i, H_i.flatten()) @ self.P[i] + self.Q_matrices[i]

class UKF_Trainer(Generic_RHONN_Trainer):
    def __init__(self, n_neurons, eta=1.0, alpha=1e-2):
        self.n_neurons = n_neurons
        self.weights = [np.random.randn(get_z_size(i))*0.1 for i in range(n_neurons)]
        self.P = [np.eye(get_z_size(i)) for i in range(n_neurons)]
        self.Q_matrices = [np.eye(get_z_size(i))*5e-3 for i in range(n_neurons)]
        self.R = 1e-6
        self.eta = eta
        self.alpha = alpha

    def update(self, x_kp1, x_k, u_k):
        for i in range(self.n_neurons):
            z = construct_z_vector(x_k, u_k, i)
            n = get_z_size(i)
            
            # Sigma params
            lambda_ = self.alpha**2 * n - n
            Wm = np.full(2*n+1, 1/(2*(n+lambda_)))
            Wc = np.copy(Wm)
            Wm[0] = lambda_/(n+lambda_)
            Wc[0] = Wm[0] + (3 - self.alpha**2)
            
            # Generate Sigmas
            try:
                L = np.linalg.cholesky((n + lambda_) * self.P[i])
            except:
                L = np.eye(n) * 0.1
                
            sigmas = np.zeros((2*n+1, n))
            sigmas[0] = self.weights[i]
            for k in range(n):
                sigmas[k+1] = self.weights[i] + L[:,k]
                sigmas[n+k+1] = self.weights[i] - L[:,k]
            
            # Transform
            Y_sigmas = np.dot(sigmas, z)
            y_mean = np.sum(Wm * Y_sigmas)
            
            # Covariances
            Py = np.sum(Wc * (Y_sigmas - y_mean)**2) + self.R
            Pxy = np.zeros(n)
            for k in range(2*n+1):
                Pxy += Wc[k] * (sigmas[k] - self.weights[i]) * (Y_sigmas[k] - y_mean)
                
            # Update
            K = Pxy / Py
            err = x_kp1[i] - y_mean
            self.weights[i] += self.eta * K * err
            self.P[i] -= np.outer(K, K) * Py
            
            # Regularize P
            self.P[i] += np.eye(n)*1e-6

class PF_Trainer(Generic_RHONN_Trainer):
    """
    Robust Particle Filter Trainer with NaN prevention strategies.
    Now supports neuron-specific Q and R values.
    """
    def __init__(self, n_neurons, n_particles=500, Q_std=None, R_std=None):
        self.n_neurons = n_neurons
        self.n_particles = n_particles
        
        # Initialize particles with controlled variance
        self.particles = [np.random.randn(n_particles, get_z_size(i)) * 0.2 
                         for i in range(n_neurons)]
        
        # Initialize weights uniformly
        self.weights_pf = [np.ones(n_particles) / n_particles for _ in range(n_neurons)]
        
        # Tuning parameters - neuron-specific or default
        if Q_std is None:
            # Default: same for all neurons
            self.Q_std = [0.05] * n_neurons
        elif isinstance(Q_std, (list, np.ndarray)):
            # Use provided neuron-specific values
            assert len(Q_std) == n_neurons, f"Q_std must have length {n_neurons}"
            self.Q_std = list(Q_std)
        else:
            # Single value for all neurons
            self.Q_std = [Q_std] * n_neurons
        
        if R_std is None:
            # Default: same for all neurons
            self.R_std = [0.05] * n_neurons
        elif isinstance(R_std, (list, np.ndarray)):
            # Use provided neuron-specific values
            assert len(R_std) == n_neurons, f"R_std must have length {n_neurons}"
            self.R_std = list(R_std)
        else:
            # Single value for all neurons
            self.R_std = [R_std] * n_neurons
        
        self.regularization_std = 0.001  # Jitter after resampling
        
        # NaN prevention parameters
        self.min_weight = 1e-300    # Minimum weight to prevent underflow
        self.max_log_likelihood = 100.0  # Clip extreme likelihoods
        self.resample_threshold = 0.5   # Resample when Neff < threshold * N
        
    def _normalize_weights(self, weights):
        """
        Safely normalize weights with NaN and underflow protection.
        """
        # Check for NaN or inf
        if np.any(~np.isfinite(weights)):
            print("⚠️  Warning: Non-finite weights detected, resetting to uniform")
            return np.ones_like(weights) / len(weights)
        
        # Ensure positive weights
        weights = np.maximum(weights, self.min_weight)
        
        # Normalize
        weight_sum = np.sum(weights)
        if weight_sum < self.min_weight or not np.isfinite(weight_sum):
            print("⚠️  Warning: Invalid weight sum, resetting to uniform")
            return np.ones_like(weights) / len(weights)
        
        return weights / weight_sum
    
    def _compute_log_likelihood(self, errors, neuron_idx):
        """
        Compute log-likelihood with numerical stability (using Gaussian).
        Uses neuron-specific R_std.
        """
        # Clip extreme errors
        errors_clipped = np.clip(errors, -1000, 1000)
        
        # Gaussian log-likelihood: -0.5 * (err/sigma)^2
        # Student-t log-likelihood with df=3
        # log p(y|x) ∝ -((df+1)/2) * log(1 + (err/sigma)^2 / df)
        df = 3.0  # degrees of freedom (heavy tails)
        sigma = self.R_std[neuron_idx]
        
        # Compute log-likelihood
        normalized_errors_sq = (errors_clipped / sigma) ** 2
        log_likelihood = -((df + 1) / 2.0) * np.log(1.0 + normalized_errors_sq / df)
        
        # Clip to prevent overflow in exp
        # log_likelihood = np.clip(log_likelihood, -self.max_log_likelihood, 0)
        
        return log_likelihood
    
    def _resample_particles(self, neuron_idx):
        """
        Systematic resampling with regularization.
        """
        weights = self.weights_pf[neuron_idx]
        particles = self.particles[neuron_idx]
        n_weights = get_z_size(neuron_idx)
        
        # Normalize weights
        weights = self._normalize_weights(weights)
        
        # Systematic resampling
        positions = (np.arange(self.n_particles) + np.random.random()) / self.n_particles
        cumulative_sum = np.cumsum(weights)
        
        indices = np.searchsorted(cumulative_sum, positions)
        indices = np.clip(indices, 0, self.n_particles - 1)
        
        # Resample particles
        self.particles[neuron_idx] = particles[indices].copy()
        
        # Add regularization jitter (preserve particle diversity)
        jitter = np.random.randn(self.n_particles, n_weights) * self.regularization_std
        self.particles[neuron_idx] += jitter
        
        # Reset weights to uniform
        self.weights_pf[neuron_idx] = np.ones(self.n_particles) / self.n_particles
    
    def update(self, x_kp1, x_k, u_k):
        """
        Update step with NaN prevention and neuron-specific Q/R.
        """
        for i in range(self.n_neurons):
            try:
                # Get feature vector
                z = construct_z_vector(x_k, u_k, i)
                n_weights = get_z_size(i)
                
                # Check for NaN in input
                if not np.all(np.isfinite(z)):
                    print(f"⚠️  Warning: Non-finite feature vector for neuron {i}, skipping update")
                    continue
                
                # 1. PREDICTION: Add process noise (drift) - neuron-specific Q_std
                drift = np.random.randn(self.n_particles, n_weights) * self.Q_std[i]
                self.particles[i] += drift
                
                # Clip particles to prevent extreme values
                self.particles[i] = np.clip(self.particles[i], -1000, 1000)
                
                # 2. UPDATE: Compute likelihoods
                preds = self.particles[i] @ z
                
                # Check for NaN in predictions
                if not np.all(np.isfinite(preds)):
                    print(f"⚠️  Warning: Non-finite predictions for neuron {i}, resetting particles")
                    self.particles[i] = np.random.randn(self.n_particles, n_weights) * 0.1
                    continue
                
                # Compute errors
                errors = x_kp1[i] - preds
                
                # Compute log-likelihood (numerically stable) - neuron-specific R_std
                log_likelihood = self._compute_log_likelihood(errors, i)
                
                # Shift for numerical stability before exp
                log_likelihood -= np.max(log_likelihood)
                
                # Update weights
                self.weights_pf[i] *= np.exp(log_likelihood)
                
                # Normalize weights
                self.weights_pf[i] = self._normalize_weights(self.weights_pf[i])
                
                # 3. RESAMPLING: Check effective sample size
                weight_sq_sum = np.sum(self.weights_pf[i] ** 2)
                if weight_sq_sum > 0:
                    eff_N = 1.0 / weight_sq_sum
                else:
                    eff_N = 0
                
                # Resample if effective sample size is too low
                if eff_N < self.resample_threshold * self.n_particles:
                    self._resample_particles(i)
                    
            except Exception as e:
                print(f"⚠️  Error in PF update for neuron {i}: {e}")
                # Reset to safe state
                self.particles[i] = np.random.randn(self.n_particles, get_z_size(i)) * 0.1
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
    
    def get_estimates(self):
        """
        Get weighted average of particles (with NaN protection).
        """
        estimates = []
        for i in range(self.n_neurons):
            # Normalize weights
            weights = self._normalize_weights(self.weights_pf[i])
            
            # Compute weighted average
            estimate = np.average(self.particles[i], axis=0, weights=weights)
            
            # Check for NaN
            if not np.all(np.isfinite(estimate)):
                print(f"⚠️  Warning: Non-finite estimate for neuron {i}, using median")
                estimate = np.median(self.particles[i], axis=0)
            
            estimates.append(estimate)
        
        return estimates

# ============================================================
# 4) Error Metrics Functions
# ============================================================
def calculate_error_metrics(y_true, y_pred, metric_name="State"):
    """
    Calculate comprehensive error metrics.
    
    Args:
        y_true: True values (n_samples, n_states)
        y_pred: Predicted values (n_samples, n_states)
        metric_name: Name for reporting
    
    Returns:
        Dictionary with error metrics
    """
    errors = y_true - y_pred
    
    # Mean Absolute Error (MAE)
    mae = np.mean(np.abs(errors), axis=0)
    mae_total = np.mean(mae)
    
    # Root Mean Square Error (RMSE)
    rmse = np.sqrt(np.mean(errors**2, axis=0))
    rmse_total = np.sqrt(np.mean(rmse**2))
    
    # Normalized RMSE (NRMSE) - normalized by range
    ranges = np.max(y_true, axis=0) - np.min(y_true, axis=0)
    ranges[ranges < 1e-10] = 1.0  # Avoid division by zero
    nrmse = rmse / ranges
    nrmse_total = np.mean(nrmse)
    
    # Maximum Absolute Error
    max_error = np.max(np.abs(errors), axis=0)
    
    # Mean Squared Error (MSE)
    mse = np.mean(errors**2, axis=0)
    mse_total = np.mean(mse)
    
    # R² Score (coefficient of determination)
    ss_res = np.sum(errors**2, axis=0)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0))**2, axis=0)
    r2 = 1 - (ss_res / (ss_tot + 1e-10))
    r2_total = np.mean(r2)
    
    return {
        'MAE': mae,
        'MAE_total': mae_total,
        'RMSE': rmse,
        'RMSE_total': rmse_total,
        'NRMSE': nrmse,
        'NRMSE_total': nrmse_total,
        'MSE': mse,
        'MSE_total': mse_total,
        'Max_Error': max_error,
        'R2': r2,
        'R2_total': r2_total
    }

In [2]:
import time  # Añadir al inicio del archivo

# ============================================================
# 5) Simulation Main Loop - DOUBLE PENDULUM
# ============================================================
if __name__ == "__main__":
    # ============================================================
    # Main Simulation - PARALLEL CONFIGURATION
    # ============================================================
    n_steps = 1500
    dt = 0.01
    t = np.linspace(0, (n_steps-1)*dt, n_steps)
    
    n_states = 4  # [θ1, θ2, ω1, ω2]
    
    # Measurement noise parameters (simulating real sensors with non-Gaussian characteristics)
    angle_noise_std = 0.01        # 0.01 rad (~0.57°) angle error
    omega_noise_std = 0.02        # 0.02 rad/s angular velocity error
    
    # Non-Gaussian noise parameters
    t_df = 3.0                    # Degrees of freedom for Student's t-distribution (heavier tails)
    mixture_outlier_prob = 0.09   # Probability of outlier (5%)
    outlier_scale = 5.0           # Outlier magnitude multiplier
    
    # ============================================================
    # GENERATE INITIAL WEIGHTS (UNIFORM DISTRIBUTION) - SHARED BY ALL FILTERS
    # ============================================================
    np.random.seed(7517)  # For reproducibility
    initial_weights = []
    for i in range(n_states):
        # Uniform distribution U(-1.0, 1.0)
        w_init = np.random.uniform(-1.0, 1.0, size=get_z_size(i))
        initial_weights.append(w_init.copy())
    
    print("Initial RHONN weights (uniform distribution):")
    for i, w in enumerate(initial_weights):
        print(f"  Neuron {i}: shape={w.shape}, min={w.min():.4f}, max={w.max():.4f}, mean={w.mean():.4f}")
    
    # ============================================================
    # PF-RHONN Neuron-Specific Q and R values
    # ============================================================
    pf_Q_std = [
        0.5,   # Neuron 0 (θ1): angle state
        0.5,   # Neuron 1 (θ2): angle state
        0.45,   # Neuron 2 (ω1): velocity state (slightly higher dynamics)
        0.4    # Neuron 3 (ω2): velocity state (slightly higher dynamics)
    ]
    
    pf_R_std = [
        1.0e-3,   # Neuron 0 (θ1): angle measurement
        1.0e-3,   # Neuron 1 (θ2): angle measurement
        1.0e-2,   # Neuron 2 (ω1): velocity measurement (noisier)
        1.0e-2    # Neuron 3 (ω2): velocity measurement (noisier)
    ]
    
    print("\n" + "="*70)
    print("PF-RHONN Neuron-Specific Parameters:")
    print("="*70)
    state_names = ['θ1 (angle 1)', 'θ2 (angle 2)', 'ω1 (ang vel 1)', 'ω2 (ang vel 2)']
    for i in range(n_states):
        print(f"  Neuron {i} ({state_names[i]}): Q_std={pf_Q_std[i]:.4f}, R_std={pf_R_std[i]:.4f}")
    
    # Initialize Trainers
    ekf = EKF_Trainer(n_states, eta=1.0, P0=1.0, Q=1e-3, R=1e-4)
    ukf = UKF_Trainer(n_states, eta=1.0, alpha=1e-2)
    pf = PF_Trainer(n_states, n_particles=1000, Q_std=pf_Q_std, R_std=pf_R_std)
    
    # Set initial weights for all filters
    for i in range(n_states):
        ekf.weights[i] = initial_weights[i].copy()
        ukf.weights[i] = initial_weights[i].copy()
        # For particle filter, initialize all particles with the same weights
        pf.particles[i] = np.tile(initial_weights[i], (pf.n_particles, 1))
    
    print("\n✓ All filters initialized with the same RHONN weights")
    
    # Arrays for states
    x_true = np.zeros((n_steps, 4))
    x_est_ekf = np.zeros((n_steps, 4))
    x_est_ukf = np.zeros((n_steps, 4))
    x_est_pf = np.zeros((n_steps, 4))
    
    # Arrays for noisy measurements
    y_measured = np.zeros((n_steps, 4))
    
    # Variables para medir tiempos de entrenamiento
    ekf_training_times = []
    ukf_training_times = []
    pf_training_times = []
    
    # Initial Conditions (double pendulum hanging down with small perturbation)
    x_true[0] = [np.pi/6, np.pi/4, 0.0, 0.0]  # [θ1, θ2, ω1, ω2]
    x_est_ekf[0] = x_true[0]
    x_est_ukf[0] = x_true[0]
    x_est_pf[0] = x_true[0]
    
    # Add non-Gaussian noise to initial measurement
    # Student's t-distribution with mixture of outliers
    initial_noise = np.array([
        np.random.standard_t(t_df) * angle_noise_std * np.sqrt((t_df-2)/t_df),
        np.random.standard_t(t_df) * angle_noise_std * np.sqrt((t_df-2)/t_df),
        np.random.standard_t(t_df) * omega_noise_std * np.sqrt((t_df-2)/t_df),
        np.random.standard_t(t_df) * omega_noise_std * np.sqrt((t_df-2)/t_df)
    ])
    y_measured[0] = x_true[0] + initial_noise
    
    # Excitation Input (external torques)
    u_hist = np.zeros((n_steps, 2))
    for k in range(n_steps):
        # Rich excitation signal with multiple frequencies
        τ1 = 0.5 * np.sin(1.0 * t[k]) + 0.3 * np.sin(2.5 * t[k])
        τ2 = 0.4 * np.sin(1.5 * t[k]) + 0.2 * np.cos(3.0 * t[k])
        
        # Add some step changes for better excitation
        if 400 < k < 450:
            τ1 += 1.0
            τ2 += 0.5
        elif 900 < k < 950:
            τ1 -= 0.8
            τ2 += 0.6
        elif 1200 < k < 1250:
            τ1 += 0.6
            τ2 -= 0.7
            
        u_hist[k] = [τ1, τ2]

    print("\n" + "="*70)
    print("Simulating Double Pendulum with PARALLEL CONFIGURATION...")
    print("="*70)
    print("\nEstructura de características por neurona:")
    for i in range(n_states):
        print(f"  Neurona {i} ({state_names[i]}): {get_z_size(i)} características")
    
    print("\nNoise Model (Non-Gaussian - Realistic Sensors):")
    print(f"  Process Noise: Laplace distribution (heavy tails)")
    print(f"  Measurement Noise: Student's t-distribution (df={t_df}) + Gaussian Mixture")
    print(f"    - Angles (θ1, θ2): ±{angle_noise_std:.4f} rad (~{np.degrees(angle_noise_std):.2f}°)")
    print(f"    - Angular velocities (ω1, ω2): ±{omega_noise_std:.4f} rad/s")
    print(f"    - Outlier probability: {mixture_outlier_prob*100:.1f}% (scale: {outlier_scale}x)")
    print(f"\nIntegration Method: Runge-Kutta 4th Order (RK4)")
    print(f"Time step (dt): {dt} s")
    print(f"Simulation duration: {n_steps*dt:.1f} s ({n_steps} steps)")
    
    # Main simulation loop with PARALLEL CONFIGURATION
    for k in range(n_steps - 1):
        # 1. Generate true next state
        x_true[k+1] = plant(x_true[k], u_hist[k], dt)
        
        # 2. Create noisy measurement with NON-GAUSSIAN noise (realistic sensors)
        # Using Student's t-distribution with Gaussian mixture for outliers
        measurement_noise = np.zeros(4)
        
        for i in range(4):
            # Determine if this is an outlier measurement
            is_outlier = np.random.random() < mixture_outlier_prob
            
            if i < 2:  # Angles (θ1, θ2)
                if is_outlier:
                    # Outlier: larger Gaussian noise
                    measurement_noise[i] = np.random.normal(0, angle_noise_std * outlier_scale)
                else:
                    # Normal: Student's t-distribution (heavy tails)
                    # Scale adjusted to match desired standard deviation
                    t_sample = np.random.standard_t(t_df)
                    measurement_noise[i] = t_sample * angle_noise_std * np.sqrt((t_df-2)/t_df)
            else:  # Angular velocities (ω1, ω2)
                if is_outlier:
                    # Outlier: larger Gaussian noise
                    measurement_noise[i] = np.random.normal(0, omega_noise_std * outlier_scale)
                else:
                    # Normal: Student's t-distribution (heavy tails)
                    t_sample = np.random.standard_t(t_df)
                    measurement_noise[i] = t_sample * omega_noise_std * np.sqrt((t_df-2)/t_df)
        
        # y_measured[k+1] = x_true[k+1] + measurement_noise
        y_measured[k+1] = x_true[k+1]
        
        # 3. Update filters with NOISY MEASUREMENTS
        # Each filter uses its OWN previous estimate as input
        
        # --- EKF Update & Predict ---
        start_time = time.perf_counter()
        ekf.update(y_measured[k+1], x_est_ekf[k], u_hist[k])
        ekf_time = time.perf_counter() - start_time
        ekf_training_times.append(ekf_time)
        
        # Predict next state using EKF's OWN estimate
        for i in range(4):
            z_ekf = construct_z_vector(x_est_ekf[k], u_hist[k], i)
            x_est_ekf[k+1, i] = np.dot(ekf.weights[i], z_ekf)
        
        # --- UKF Update & Predict ---
        start_time = time.perf_counter()
        ukf.update(y_measured[k+1], x_est_ukf[k], u_hist[k])
        ukf_time = time.perf_counter() - start_time
        ukf_training_times.append(ukf_time)
        
        # Predict next state using UKF's OWN estimate
        for i in range(4):
            z_ukf = construct_z_vector(x_est_ukf[k], u_hist[k], i)
            x_est_ukf[k+1, i] = np.dot(ukf.weights[i], z_ukf)
        
        # --- PF Update & Predict ---
        start_time = time.perf_counter()
        pf.update(y_measured[k+1], x_est_pf[k], u_hist[k])
        pf_time = time.perf_counter() - start_time
        pf_training_times.append(pf_time)
        
        w_pf = pf.get_estimates()
        # Predict next state using PF's OWN estimate
        for i in range(4):
            z_pf = construct_z_vector(x_est_pf[k], u_hist[k], i)
            x_est_pf[k+1, i] = np.dot(w_pf[i], z_pf)
        
        # Progress reporting
        if k % 300 == 0 and k > 0:
            print(f"Step {k}/{n_steps-1}")
            # Show current errors
            err_ekf = np.linalg.norm(x_true[k] - x_est_ekf[k])
            err_ukf = np.linalg.norm(x_true[k] - x_est_ukf[k])
            err_pf = np.linalg.norm(x_true[k] - x_est_pf[k])
            print(f"  Current errors - EKF: {err_ekf:.4f}, UKF: {err_ukf:.4f}, PF: {err_pf:.4f}")

    # ============================================================
    # 6) Calculate Comprehensive Error Metrics
    # ============================================================
    
    print("\n" + "="*70)
    print("📊 COMPREHENSIVE RESULTS - PARALLEL CONFIGURATION WITH NON-GAUSSIAN NOISE")
    print("="*70)
    
    # Calculate comprehensive error metrics for each filter
    metrics_ekf = calculate_error_metrics(x_true, x_est_ekf, "EKF-RHONN")
    metrics_ukf = calculate_error_metrics(x_true, x_est_ukf, "UKF-RHONN")
    metrics_pf = calculate_error_metrics(x_true, x_est_pf, "PF-RHONN")
    
    # Print comprehensive metrics
    print("\n--- EKF-RHONN Performance Metrics ---")
    print(f"RMSE total: {metrics_ekf['RMSE_total']:.6f}")
    for i, name in enumerate(state_names):
        print(f"  {name}: RMSE={metrics_ekf['RMSE'][i]:.6f}, MAE={metrics_ekf['MAE'][i]:.6f}, "
              f"NRMSE={metrics_ekf['NRMSE'][i]:.4f}, R²={metrics_ekf['R2'][i]:.4f}")
    
    print("\n--- UKF-RHONN Performance Metrics ---")
    print(f"RMSE total: {metrics_ukf['RMSE_total']:.6f}")
    for i, name in enumerate(state_names):
        print(f"  {name}: RMSE={metrics_ukf['RMSE'][i]:.6f}, MAE={metrics_ukf['MAE'][i]:.6f}, "
              f"NRMSE={metrics_ukf['NRMSE'][i]:.4f}, R²={metrics_ukf['R2'][i]:.4f}")
    
    print("\n--- PF-RHONN Performance Metrics ---")
    print(f"RMSE total: {metrics_pf['RMSE_total']:.6f}")
    for i, name in enumerate(state_names):
        print(f"  {name}: RMSE={metrics_pf['RMSE'][i]:.6f}, MAE={metrics_pf['MAE'][i]:.6f}, "
              f"NRMSE={metrics_pf['NRMSE'][i]:.4f}, R²={metrics_pf['R2'][i]:.4f}")
    
    # Análisis de tiempos de entrenamiento
    print("\n" + "="*70)
    print("⏱️  TIEMPOS DE ENTRENAMIENTO POR FILTRO")
    print("="*70)
    
    # Calcular estadísticas de tiempos
    ekf_total_time = np.sum(ekf_training_times)
    ukf_total_time = np.sum(ukf_training_times)
    pf_total_time = np.sum(pf_training_times)
    
    ekf_mean_time = np.mean(ekf_training_times)
    ukf_mean_time = np.mean(ukf_training_times)
    pf_mean_time = np.mean(pf_training_times)
    
    ekf_std_time = np.std(ekf_training_times)
    ukf_std_time = np.std(ukf_training_times)
    pf_std_time = np.std(pf_training_times)
    
    print(f"\n📈 Estadísticas de Tiempos por Iteración (en segundos):")
    print(f"\nEKF-RHONN:")
    print(f"  Total: {ekf_total_time:.6f} s | Media: {ekf_mean_time*1000:.4f} ms")
    print(f"  Std: {ekf_std_time*1000:.4f} ms | Min: {np.min(ekf_training_times)*1000:.4f} ms")
    print(f"  Max: {np.max(ekf_training_times)*1000:.4f} ms")
    
    print(f"\nUKF-RHONN:")
    print(f"  Total: {ukf_total_time:.6f} s | Media: {ukf_mean_time*1000:.4f} ms")
    print(f"  Std: {ukf_std_time*1000:.4f} ms | Min: {np.min(ukf_training_times)*1000:.4f} ms")
    print(f"  Max: {np.max(ukf_training_times)*1000:.4f} ms")
    
    print(f"\nPF-RHONN:")
    print(f"  Total: {pf_total_time:.6f} s | Media: {pf_mean_time*1000:.4f} ms")
    print(f"  Std: {pf_std_time*1000:.4f} ms | Min: {np.min(pf_training_times)*1000:.4f} ms")
    print(f"  Max: {np.max(pf_training_times)*1000:.4f} ms")
    
    # Comparación relativa
    print(f"\n⚡ Comparación Relativa de Velocidad:")
    print(f"  EKF es {ukf_mean_time/ekf_mean_time:.2f}x más rápido que UKF")
    print(f"  EKF es {pf_mean_time/ekf_mean_time:.2f}x más rápido que PF")
    print(f"  UKF es {pf_mean_time/ukf_mean_time:.2f}x más rápido que PF")
    
    # Calcular eficiencia (precisión por unidad de tiempo)
    print(f"\n🎯 Eficiencia (R² / Tiempo de Entrenamiento):")
    print(f"  EKF: {metrics_ekf['R2_total']/ekf_total_time:.4f} R²/s")
    print(f"  UKF: {metrics_ukf['R2_total']/ukf_total_time:.4f} R²/s")
    print(f"  PF:  {metrics_pf['R2_total']/pf_total_time:.4f} R²/s")
    
    # Analysis of noise characteristics
    print("\n" + "="*70)
    print("📉 ANÁLISIS DE RUIDO (Non-Gaussian)")
    print("="*70)
    
    # Calculate measurement noise statistics
    meas_noise = y_measured - x_true
    print(f"\nCaracterísticas del ruido de medición:")
    for i, name in enumerate(state_names):
        noise_samples = meas_noise[:, i]
        kurtosis = np.mean((noise_samples - np.mean(noise_samples))**4) / (np.std(noise_samples)**4)
        skewness = np.mean((noise_samples - np.mean(noise_samples))**3) / (np.std(noise_samples)**3)
        print(f"\n  {name}:")
        print(f"    Std: {np.std(noise_samples):.6f}")
        print(f"    Kurtosis: {kurtosis:.4f} (Gaussian=3.0, higher=heavier tails)")
        print(f"    Skewness: {skewness:.4f} (Gaussian=0.0)")
        print(f"    Max abs: {np.max(np.abs(noise_samples)):.6f}")
    
    print("\n" + "="*70)
    print("🏆 MEJOR FILTRO: ", end="")
    rmse_dict = {'EKF-RHONN': metrics_ekf['RMSE_total'], 
                 'UKF-RHONN': metrics_ukf['RMSE_total'], 
                 'PF-RHONN': metrics_pf['RMSE_total']}
    best_filter = min(rmse_dict, key=rmse_dict.get)
    print(f"{best_filter} (RMSE total: {rmse_dict[best_filter]:.6f})")
    print("="*70)

Initial RHONN weights (uniform distribution):
  Neuron 0: shape=(3,), min=-0.7101, max=0.3165, mean=-0.0699
  Neuron 1: shape=(4,), min=-0.6689, max=0.9859, mean=-0.2045
  Neuron 2: shape=(4,), min=-0.7763, max=0.7724, mean=-0.2666
  Neuron 3: shape=(4,), min=-0.8238, max=0.7480, mean=-0.0612

PF-RHONN Neuron-Specific Parameters:
  Neuron 0 (θ1 (angle 1)): Q_std=0.5000, R_std=0.0010
  Neuron 1 (θ2 (angle 2)): Q_std=0.5000, R_std=0.0010
  Neuron 2 (ω1 (ang vel 1)): Q_std=0.4500, R_std=0.0100
  Neuron 3 (ω2 (ang vel 2)): Q_std=0.4000, R_std=0.0100

✓ All filters initialized with the same RHONN weights

Simulating Double Pendulum with PARALLEL CONFIGURATION...

Estructura de características por neurona:
  Neurona 0 (θ1 (angle 1)): 3 características
  Neurona 1 (θ2 (angle 2)): 4 características
  Neurona 2 (ω1 (ang vel 1)): 4 características
  Neurona 3 (ω2 (ang vel 2)): 4 características

Noise Model (Non-Gaussian - Realistic Sensors):
  Process Noise: Laplace distribution (heavy tails)
 

In [3]:
# ============================================================
# 7) Visualization - Double Pendulum
# ============================================================

print("\nGenerando visualizaciones...")

# Configuración de formato para tesis
thesis_config = {
    'font_family': 'Computer Modern, serif',
    'font_size': 14,
    'title_font_size': 16,
    'legend_font_size': 12,
    'line_width_true': 2.5,
    'line_width_est': 2.0,
    'line_width_meas': 1.0,
    'plot_width': 1000,
    'plot_height': 500,
    'grid_color': 'rgba(200, 200, 200, 0.3)',
    'grid_width': 0.5
}

# --- Gráfica de tiempos de entrenamiento ---
fig_times = go.Figure()

# Crear arrays de tiempo de simulación para el eje x
sim_time = t[1:]  # Excluir el primer tiempo (k=0)

# Agregar trazas de tiempo para cada filtro
fig_times.add_trace(go.Scatter(
    x=sim_time, y=ekf_training_times,
    mode='lines',
    name='EKF-RHONN',
    line=dict(color='#1f77b4', width=2),
))

fig_times.add_trace(go.Scatter(
    x=sim_time, y=ukf_training_times,
    mode='lines',
    name='UKF-RHONN',
    line=dict(color='#2ca02c', width=2),
))

fig_times.add_trace(go.Scatter(
    x=sim_time, y=pf_training_times,
    mode='lines',
    name='PF-RHONN',
    line=dict(color='#d62728', width=2),
))

fig_times.update_layout(
    title={
        'text': 'Tiempos de Entrenamiento por Iteración',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo de Simulación (s)',
    yaxis_title='Tiempo de Entrenamiento (s)',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        type='log',  # Escala logarítmica para mejor visualización
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_times.show()

# --- Gráfica de barras comparando tiempos medios ---
fig_times_bar = go.Figure()

filters = ['EKF-RHONN', 'UKF-RHONN', 'PF-RHONN']
mean_times_ms = [ekf_mean_time*1000, ukf_mean_time*1000, pf_mean_time*1000]

fig_times_bar.add_trace(go.Bar(
    x=filters,
    y=mean_times_ms,
    marker_color=['#1f77b4', '#2ca02c', '#d62728'],
    text=[f'{t:.3f} ms' for t in mean_times_ms],
    textposition='outside'
))

fig_times_bar.update_layout(
    title={
        'text': 'Tiempo Promedio de Entrenamiento por Iteración',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tipo de Filtro',
    yaxis_title='Tiempo Promedio (ms)',
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    xaxis=dict(
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_times_bar.show()

# --- Visualización de la trayectoria del doble péndulo (animación conceptual) ---
# Calcular posiciones cartesianas de las masas del péndulo
l1, l2 = 1.0, 1.2  # longitudes de los péndulos

x1_true = l1 * np.sin(x_true[:, 0])
y1_true = -l1 * np.cos(x_true[:, 0])
x2_true = x1_true + l2 * np.sin(x_true[:, 1])
y2_true = y1_true - l2 * np.cos(x_true[:, 1])

# Trayectoria de la segunda masa (la más interesante)
fig_traj = go.Figure()

fig_traj.add_trace(go.Scatter(
    x=x2_true, y=y2_true,
    mode='lines',
    name='Trayectoria Real (Masa 2)',
    line=dict(color='#000000', width=2),
))

# Puntos de inicio y final
fig_traj.add_trace(go.Scatter(
    x=[x2_true[0]], y=[y2_true[0]],
    mode='markers',
    name='Inicio',
    marker=dict(size=12, color='green', symbol='circle'),
))

fig_traj.add_trace(go.Scatter(
    x=[x2_true[-1]], y=[y2_true[-1]],
    mode='markers',
    name='Final',
    marker=dict(size=12, color='red', symbol='square'),
))

fig_traj.update_layout(
    title={
        'text': 'Trayectoria del Doble Péndulo (Masa 2)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Posición x (m)',
    yaxis_title='Posición y (m)',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5,
    ),
    yaxis=dict(
        scaleanchor="x",
        scaleratio=1,
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_width'],  # Cuadrado
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_traj.show()

# --- Gráficas por Estado (con medidas ruidosas) ---
states_info = [
    {'idx': 0, 'var': 'θ₁', 'desc': 'Ángulo del Primer Péndulo', 'y_label': 'Ángulo θ₁ (rad)'},
    {'idx': 1, 'var': 'θ₂', 'desc': 'Ángulo del Segundo Péndulo', 'y_label': 'Ángulo θ₂ (rad)'},
    {'idx': 2, 'var': 'ω₁', 'desc': 'Velocidad Angular 1', 'y_label': 'Velocidad ω₁ (rad/s)'},
    {'idx': 3, 'var': 'ω₂', 'desc': 'Velocidad Angular 2', 'y_label': 'Velocidad ω₂ (rad/s)'}
]

for state_info in states_info:
    i = state_info['idx']
    
    fig = go.Figure()
    
    # True state
    fig.add_trace(go.Scatter(
        x=t, y=x_true[:, i],
        mode='lines',
        name='Estado Real',
        line=dict(color='#000000', width=thesis_config['line_width_true']),
        showlegend=True
    ))
    
    # Noisy measurements
    fig.add_trace(go.Scatter(
        x=t, y=y_measured[:, i],
        mode='markers',
        name='Medidas',
        marker=dict(size=2.5, color='rgba(138, 43, 226, 0.6)'),
        showlegend=True
    ))
    
    # EKF estimate
    fig.add_trace(go.Scatter(
        x=t, y=x_est_ekf[:, i],
        mode='lines',
        name='EKF-RHONN',
        line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
        showlegend=True
    ))
    
    # UKF estimate
    fig.add_trace(go.Scatter(
        x=t, y=x_est_ukf[:, i],
        mode='lines',
        name='UKF-RHONN',
        line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
        showlegend=True
    ))
    
    # PF estimate
    fig.add_trace(go.Scatter(
        x=t, y=x_est_pf[:, i],
        mode='lines',
        name='PF-RHONN',
        line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
        showlegend=True
    ))
    
    fig.update_layout(
        title={
            'text': f'Estado {state_info["var"]}: {state_info["desc"]}',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title=state_info['y_label'],
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig.show()

# --- Gráfica de errores acumulados ---
fig_errors = go.Figure()

# Calculate cumulative errors over time
cumulative_error_ekf = np.zeros(n_steps)
cumulative_error_ukf = np.zeros(n_steps)
cumulative_error_pf = np.zeros(n_steps)

for k in range(1, n_steps):
    error_ekf = np.linalg.norm(x_true[k] - x_est_ekf[k])
    error_ukf = np.linalg.norm(x_true[k] - x_est_ukf[k])
    error_pf = np.linalg.norm(x_true[k] - x_est_pf[k])
    
    cumulative_error_ekf[k] = cumulative_error_ekf[k-1] + error_ekf
    cumulative_error_ukf[k] = cumulative_error_ukf[k-1] + error_ukf
    cumulative_error_pf[k] = cumulative_error_pf[k-1] + error_pf

fig_errors.add_trace(go.Scatter(
    x=t, y=cumulative_error_ekf,
    mode='lines',
    name='EKF-RHONN',
    line=dict(color='#1f77b4', width=2),
))

fig_errors.add_trace(go.Scatter(
    x=t, y=cumulative_error_ukf,
    mode='lines',
    name='UKF-RHONN',
    line=dict(color='#2ca02c', width=2),
))

fig_errors.add_trace(go.Scatter(
    x=t, y=cumulative_error_pf,
    mode='lines',
    name='PF-RHONN',
    line=dict(color='#d62728', width=2),
))

fig_errors.update_layout(
    title={
        'text': 'Error Acumulado en el Tiempo - Norma Euclidiana',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo (s)',
    yaxis_title='Error Acumulado',
    xaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_errors.show()

# --- Gráfica de compensación tiempo-precision (Pareto) ---
fig_pareto = go.Figure()

fig_pareto.add_trace(go.Scatter(
    x=[ekf_mean_time*1000, ukf_mean_time*1000, pf_mean_time*1000],
    y=[metrics_ekf['RMSE_total'], metrics_ukf['RMSE_total'], metrics_pf['RMSE_total']],
    mode='markers+text',
    text=['EKF', 'UKF', 'PF'],
    textposition='top center',
    marker=dict(
        size=15,
        color=['#1f77b4', '#2ca02c', '#d62728'],
        line=dict(width=2, color='DarkSlateGrey')
    ),
    name='Filtros'
))

fig_pareto.update_layout(
    title={
        'text': 'Compensación Tiempo-Precisión (Frente de Pareto)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis_title='Tiempo Promedio por Iteración (ms)',
    yaxis_title='Error Cuadrático Medio (RMSE)',
    xaxis=dict(
        type='log',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    yaxis=dict(
        type='log',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60)
)

fig_pareto.show()

# --- Gráfica de comparación de métricas ---
fig_metrics = go.Figure()

metrics_names = ['RMSE', 'MAE', 'NRMSE']
ekf_vals = [metrics_ekf['RMSE_total'], metrics_ekf['MAE_total'], metrics_ekf['NRMSE_total']]
ukf_vals = [metrics_ukf['RMSE_total'], metrics_ukf['MAE_total'], metrics_ukf['NRMSE_total']]
pf_vals = [metrics_pf['RMSE_total'], metrics_pf['MAE_total'], metrics_pf['NRMSE_total']]

x = np.arange(len(metrics_names))
width = 0.25

fig_metrics.add_trace(go.Bar(
    x=x - width, y=ekf_vals, width=width,
    name='EKF-RHONN',
    marker_color='#1f77b4'
))

fig_metrics.add_trace(go.Bar(
    x=x, y=ukf_vals, width=width,
    name='UKF-RHONN',
    marker_color='#2ca02c'
))

fig_metrics.add_trace(go.Bar(
    x=x + width, y=pf_vals, width=width,
    name='PF-RHONN',
    marker_color='#d62728'
))

fig_metrics.update_layout(
    title={
        'text': 'Comparación de Métricas de Error',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
    },
    xaxis=dict(
        tickmode='array',
        tickvals=x,
        ticktext=metrics_names,
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        title='Valor de la Métrica',
        showgrid=True,
        gridcolor=thesis_config['grid_color'],
        gridwidth=thesis_config['grid_width'],
        showline=True,
        linewidth=1.5,
        linecolor='black',
        mirror=True,
        ticks='outside',
        tickwidth=1.5,
        ticklen=5
    ),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
    ),
    font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
    plot_bgcolor='white',
    paper_bgcolor='white',
    width=thesis_config['plot_width'],
    height=thesis_config['plot_height'],
    margin=dict(l=80, r=40, t=80, b=60),
    barmode='group'
)

fig_metrics.show()

print("\n✅ Visualización completa con configuración paralela y análisis de métricas.")

# ============================================================
# 8) Additional Analysis
# ============================================================

print("\n" + "="*70)
print("📈 ANÁLISIS ADICIONAL")
print("="*70)

# Analyze convergence
convergence_window = 100  # Look at last 100 steps
if n_steps > convergence_window:
    final_metrics = {
        'EKF': calculate_error_metrics(x_true[-convergence_window:, :], 
                                       x_est_ekf[-convergence_window:, :]),
        'UKF': calculate_error_metrics(x_true[-convergence_window:, :], 
                                       x_est_ukf[-convergence_window:, :]),
        'PF': calculate_error_metrics(x_true[-convergence_window:, :], 
                                      x_est_pf[-convergence_window:, :])
    }
    
    print(f"\nMétricas en los últimos {convergence_window} pasos (estado estacionario):")
    for filter_name, metrics in final_metrics.items():
        print(f"\n{filter_name}:")
        print(f"  RMSE total: {metrics['RMSE_total']:.6f}")
        for i, name in enumerate(state_names):
            print(f"    {name}: RMSE={metrics['RMSE'][i]:.6f}, R²={metrics['R2'][i]:.4f}")

# Resumen final
print("\n" + "="*70)
print("🎯 RESUMEN EJECUTIVO")
print("="*70)
print(f"Filtro más preciso (menor RMSE): {best_filter}")
print(f"Filtro más rápido: {'EKF-RHONN' if ekf_mean_time == min([ekf_mean_time, ukf_mean_time, pf_mean_time]) else 'UKF-RHONN' if ukf_mean_time == min([ekf_mean_time, ukf_mean_time, pf_mean_time]) else 'PF-RHONN'}")
print(f"\nRecomendaciones:")
print("1. Para aplicaciones en tiempo real: EKF-RHONN (velocidad)")
print(f"2. Para máxima precisión: {best_filter}")
print("3. Para robustez frente a no-linealidades fuertes: UKF-RHONN o PF-RHONN")
print("\nIntegration Method Used: Runge-Kutta 4th Order (RK4) - High accuracy discretization")
print("Error Metrics: RMSE, MAE, NRMSE, R², Max Error")



Generando visualizaciones...



✅ Visualización completa con configuración paralela y análisis de métricas.

📈 ANÁLISIS ADICIONAL

Métricas en los últimos 100 pasos (estado estacionario):

EKF:
  RMSE total: 0.003977
    θ1 (angle 1): RMSE=0.007118, R²=0.9983
    θ2 (angle 2): RMSE=0.001256, R²=0.9999
    ω1 (ang vel 1): RMSE=0.003219, R²=1.0000
    ω2 (ang vel 2): RMSE=0.000818, R²=1.0000

UKF:
  RMSE total: 0.017710
    θ1 (angle 1): RMSE=0.029386, R²=0.9714
    θ2 (angle 2): RMSE=0.006867, R²=0.9975
    ω1 (ang vel 1): RMSE=0.018059, R²=0.9997
    ω2 (ang vel 2): RMSE=0.004219, R²=0.9997

PF:
  RMSE total: 0.000883
    θ1 (angle 1): RMSE=0.000342, R²=1.0000
    θ2 (angle 2): RMSE=0.000383, R²=1.0000
    ω1 (ang vel 1): RMSE=0.001221, R²=1.0000
    ω2 (ang vel 2): RMSE=0.001168, R²=1.0000

🎯 RESUMEN EJECUTIVO
Filtro más preciso (menor RMSE): PF-RHONN
Filtro más rápido: EKF-RHONN

Recomendaciones:
1. Para aplicaciones en tiempo real: EKF-RHONN (velocidad)
2. Para máxima precisión: PF-RHONN
3. Para robustez frente a

In [4]:
# Imprimir pesos finales de cada filtro
print("\n" + "="*70)
print("🔍 PESOS FINALES DE LAS REDES NEURONALES")
print("="*70)

print("\n--- EKF-RHONN ---")
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {ekf.weights[i]}")
    print(f"  Norm: {np.linalg.norm(ekf.weights[i]):.4f}, Mean: {np.mean(ekf.weights[i]):.4f}, Std: {np.std(ekf.weights[i]):.4f}")

print("\n--- UKF-RHONN ---")
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {ukf.weights[i]}")
    print(f"  Norm: {np.linalg.norm(ukf.weights[i]):.4f}, Mean: {np.mean(ukf.weights[i]):.4f}, Std: {np.std(ukf.weights[i]):.4f}")

print("\n--- PF-RHONN ---")
w_pf_final = pf.get_estimates()
for i in range(n_states):
    print(f"\nNeurona {i} ({state_names[i]}) - {get_z_size(i)} pesos:")
    print(f"  {w_pf_final[i]}")
    print(f"  Norm: {np.linalg.norm(w_pf_final[i]):.4f}, Mean: {np.mean(w_pf_final[i]):.4f}, Std: {np.std(w_pf_final[i]):.4f}")

print("\n" + "="*70)
print("✅ SIMULACIÓN COMPLETADA - DOBLE PÉNDULO")
print("="*70)
print("\nSistema: Doble Péndulo con Amortiguamiento")
print(f"Método de discretización: Runge-Kutta 4° Orden (RK4)")
print(f"Paso de tiempo: {dt} s")
print(f"Duración: {n_steps*dt:.1f} s")
print(f"Número de estados: {n_states}")
print("\nMétricas de error utilizadas:")
print("  - RMSE (Root Mean Square Error)")
print("  - MAE (Mean Absolute Error)")
print("  - NRMSE (Normalized Root Mean Square Error)")
print("  - R² (Coefficient of Determination)")
print("  - Max Error (Maximum Absolute Error)")



🔍 PESOS FINALES DE LAS REDES NEURONALES

--- EKF-RHONN ---

Neurona 0 (θ1 (angle 1)) - 3 pesos:
  [ 2.55030587  0.35502334 -3.70121419]
  Norm: 4.5088, Mean: -0.2653, Std: 2.5896

Neurona 1 (θ2 (angle 2)) - 4 pesos:
  [-3.82310013  0.54697146  6.97395358 -0.0185558 ]
  Norm: 7.9719, Mean: 0.9198, Std: 3.8784

Neurona 2 (ω1 (ang vel 1)) - 4 pesos:
  [-1.5944627   6.23746507 -2.02200876 -0.08160855]
  Norm: 6.7486, Mean: 0.6348, Std: 3.3140

Neurona 3 (ω2 (ang vel 2)) - 4 pesos:
  [ 0.63538159  4.80886901 -2.32436872 -1.06520646]
  Norm: 5.4833, Mean: 0.5137, Std: 2.6931

--- UKF-RHONN ---

Neurona 0 (θ1 (angle 1)) - 3 pesos:
  [ 2.48214487  0.29917137 -3.68108909]
  Norm: 4.4498, Mean: -0.2999, Std: 2.5515

Neurona 1 (θ2 (angle 2)) - 4 pesos:
  [-3.99803903e+00  4.70352930e-01  7.53248355e+00 -6.66545293e-03]
  Norm: 8.5407, Mean: 0.9995, Std: 4.1517

Neurona 2 (ω1 (ang vel 1)) - 4 pesos:
  [-1.48794062  6.07434389 -2.10897094  0.06387344]
  Norm: 6.6003, Mean: 0.6353, Std: 3.2384

Neu